In [1]:
from sklearn.datasets import load_sample_images
import tensorflow as tf

images = load_sample_images()["images"]

images = tf.keras.layers.CenterCrop(height=70, width=120)(images)
images = tf.keras.layers.Rescaling(scale=1 / 255)(images)


In [2]:
images.shape

TensorShape([2, 70, 120, 3])

In [3]:
conv_layer = tf.keras.layers.Conv2D(filters=32, kernel_size=7)
fmaps = conv_layer(images)


In [4]:
fmaps.shape

TensorShape([2, 64, 114, 32])

In [5]:
conv_layer = tf.keras.layers.Conv2D(filters=32, kernel_size=7, padding="same")
fmaps = conv_layer(images)
fmaps.shape

TensorShape([2, 70, 120, 32])

In [6]:
kernels, biases = conv_layer.get_weights()
kernels.shape

(7, 7, 3, 32)

In [7]:
biases.shape

(32,)

In [8]:
max_pool = tf.keras.layers.MaxPool2D(pool_size=2)

In [9]:
class DepthPool(tf.keras.layers.Layer):
    
    def __init__(self, pool_size=2, **kwargs):
        super().__init__(**kwargs)
        self.pool_size = pool_size
    
    def call(self, inputs):
        shape = tf.shape(inputs) # shape[-1] is the number of channels
        groups = shape[-1] // self.pool_size # number of channel groups
        new_shape = tf.concat([shape[:-1], [groups, self.pool_size]], axis=0)
        return tf.reduce_max(tf.reshape(inputs, new_shape), axis=-1)


In [10]:
global_avg_pool = tf.keras.layers.GlobalAvgPool2D()

In [11]:
from functools import partial

DefaultConv2D = partial(tf.keras.layers.Conv2D, kernel_size=3, padding="same",
 activation="relu", kernel_initializer="he_normal")

model = tf.keras.Sequential([
 DefaultConv2D(filters=64, kernel_size=7, input_shape=[28, 28, 1]),
 tf.keras.layers.MaxPool2D(),
 DefaultConv2D(filters=128),
 DefaultConv2D(filters=128),
 tf.keras.layers.MaxPool2D(),
 DefaultConv2D(filters=256),
 DefaultConv2D(filters=256),
 tf.keras.layers.MaxPool2D(),
 
 tf.keras.layers.Flatten(),
 tf.keras.layers.Dense(units=128, activation="relu",
 kernel_initializer="he_normal"),
 tf.keras.layers.Dropout(0.5),
 tf.keras.layers.Dense(units=64, activation="relu",
 kernel_initializer="he_normal"),
 tf.keras.layers.Dropout(0.5),
 tf.keras.layers.Dense(units=10, activation="softmax")
])


c:\Users\PP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
DefaultConv2D = partial(tf.keras.layers.Conv2D, kernel_size=3, strides=1,
 padding="same", kernel_initializer="he_normal",
 use_bias=False)

class ResidualUnit(tf.keras.layers.Layer):
    def __init__(self, filters, strides=1, activation="relu", **kwargs):
        super().__init__(**kwargs)
        self.activation = tf.keras.activations.get(activation)
        self.main_layers = [
        DefaultConv2D(filters, strides=strides),
        tf.keras.layers.BatchNormalization(),
        self.activation,
        DefaultConv2D(filters),
        tf.keras.layers.BatchNormalization()
        ]
        self.skip_layers = []
        if strides > 1:
            self.skip_layers = [
            DefaultConv2D(filters, kernel_size=1, strides=strides),
            tf.keras.layers.BatchNormalization()
            ]
 
    def call(self, inputs):
        Z = inputs
        for layer in self.main_layers:
            Z = layer(Z)
        skip_Z = inputs
        for layer in self.skip_layers:
            skip_Z = layer(skip_Z)
        return self.activation(Z + skip_Z)


In [14]:
model = tf.keras.Sequential([
 DefaultConv2D(64, kernel_size=7, strides=2, input_shape=[224, 224, 3]),
 tf.keras.layers.BatchNormalization(),
 tf.keras.layers.Activation("relu"),
 tf.keras.layers.MaxPool2D(pool_size=3, strides=2, padding="same"),
])

prev_filters = 64

for filters in [64] * 3 + [128] * 4 + [256] * 6 + [512] * 3:
    strides = 1 if filters == prev_filters else 2
    model.add(ResidualUnit(filters, strides=strides))
    prev_filters = filters

model.add(tf.keras.layers.GlobalAvgPool2D())
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(10, activation="softmax"))


In [15]:
model = tf.keras.applications.ResNet50(weights="imagenet")


102967424/102967424 ━━━━━━━━━━━━━━━━━━━━ 51s 1us/step


In [16]:
images = load_sample_images()["images"]
images_resized = tf.keras.layers.Resizing(height=224, width=224,
 crop_to_aspect_ratio=True)(images)

In [17]:
inputs = tf.keras.applications.resnet50.preprocess_input(images_resized)
Y_proba = model.predict(inputs)
Y_proba.shape

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


(2, 1000)

In [18]:
import tensorflow_datasets as tfds

dataset, info = tfds.load("tf_flowers", as_supervised=True, with_info=True)
dataset_size = info.splits["train"].num_examples # 3670
class_names = info.features["label"].names # ["dandelion", "daisy", ...]
n_classes = info.features["label"].num_classes # 5

ModuleNotFoundError: No module named 'tensorflow_datasets'

In [19]:
base_model = tf.keras.applications.xception.Xception(weights="imagenet",
 include_top=False)
avg = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
class_output = tf.keras.layers.Dense(n_classes, activation="softmax")(avg)
loc_output = tf.keras.layers.Dense(4)(avg)
model = tf.keras.Model(inputs=base_model.input,
 outputs=[class_output, loc_output])
model.compile(loss=["sparse_categorical_crossentropy", "mse"],
 loss_weights=[0.8, 0.2], # depends on what you care most about
 optimizer=optimizer, metrics=["accuracy"])


83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 46s 1us/step


NameError: name 'n_classes' is not defined